# 4. Ray tracing a thin accretion disk

Backward ray tracing: from each pixel of a distant camera we shoot a null geodesic toward the black hole and record where (if at all) it crosses the equatorial plane. The crossing radius is mapped to a temperature with a numerically integrated **Novikov-Thorne** profile, and the intensity is boosted by the Doppler factor $g^4$ of the orbiting gas.

The camera sits at 200 M, elevated by $\theta_{obs}$ above the disk plane; `(by, bz)` are the impact parameters of each pixel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from geodesics import (Schwarzschild, Kerr, solve_geodesic, cross_equator, camera_ray,
                       disk_flux, temperature_interpolator, render_image, show_image)

## One photon, equatorial crossings as `solve_ivp` events

In [ ]:
schw = Schwarzschild(dim=4, M_val=0.75)
x0, v0 = schw.from_cartesian(x=10.0, y=4.0, z=2.0, vx=-0.5, vy=0.0, vz=-1.0, massive=False)
sol = solve_geodesic(schw, x0, v0, (0, 1000), extra_events=[cross_equator])

# event 0 = horizon, event 1 = cross_equator
print(f"equatorial crossings: {len(sol.t_events[1])}")
for k, (t_c, y_c) in enumerate(zip(sol.t_events[1], sol.y_events[1])):
    print(f"  #{k}: tau={t_c:.2f}, r={y_c[1]:.4f}, phi={y_c[3]:.4f}")

## Temperature profile

$$F(r) = -\frac{\dot M}{4\pi\sqrt{-g}}\frac{\Omega_{,r}}{(E-\Omega L)^2}\int_{r_{isco}}^{r}(E-\Omega L)L_{,r}\,dr, \qquad T \propto F^{1/4}$$

$E, L, \Omega$ are computed for circular orbits from the metric components at $\theta = \pi/2$, so the same code works for Kerr. The profile is tabulated once and interpolated (thousands of geodesics would otherwise mean thousands of integrals).

In [ ]:
schw = Schwarzschild(M_val=1.0)
r = np.linspace(schw.r_isco + 0.01, 30, 3000)
plt.plot(r, disk_flux(schw, r), label='Schwarzschild')
kerr = Kerr(M_val=1.0, a_val=0.75)
r = np.linspace(kerr.r_isco + 0.01, 30, 3000)
plt.plot(r, disk_flux(kerr, r), label='Kerr a=0.75')
plt.xlabel('r'); plt.ylabel('T / T_max'); plt.legend(); plt.show()

## Rendering

Brute force: one geodesic per pixel. `N=100` is a few minutes on a laptop; the images in the README were made with `N=350`. Notebook 5 replaces this loop with a neural network.

In [ ]:
T_schw = temperature_interpolator(schw, r_max=30)
img, r_map, phi_map = render_image(schw, T_schw, theta_obs_deg=17, N=100, b_max=30)

In [ ]:
show_image(img, b_max=30, title='Schwarzschild, 17°', mode='log')

In [ ]:
show_image(img, b_max=30, title='Schwarzschild, 17° (gamma 0.3)', mode='gamma', gamma=0.3)

Nearly edge-on, the far side of the disk is lensed over and under the shadow:

In [ ]:
img_edge, _, _ = render_image(schw, T_schw, theta_obs_deg=1, N=100, b_max=30)
show_image(img_edge, b_max=30, title='Schwarzschild, 1°')

## Kerr

In [ ]:
T_kerr = temperature_interpolator(kerr, r_max=30)
img_k, _, _ = render_image(kerr, T_kerr, theta_obs_deg=17, N=100, b_max=25)
show_image(img_k, b_max=25, title='Kerr a=0.75, 17°', mode='log')